In [1]:
from src.utils.sample_data import get_sample_grad_rate_data
import pandas as pd

In [2]:
grad_rate_df = get_sample_grad_rate_data()
grad_rate_df.head()

,year,district,subgroup,metric,value
0,2020,Canidae School District,All Students,Four-Year Graduation Rate,65.103827
1,2020,Canidae School District,All Students,Five-Year Graduation Rate,68.359018
2,2020,Canidae School District,All Students,Six-Year Graduation Rate,71.776969
3,2021,Canidae School District,All Students,Four-Year Graduation Rate,65.184729
4,2021,Canidae School District,All Students,Five-Year Graduation Rate,68.443965


In [10]:
%%time

# Filter data
grad_rate_filtered_df = grad_rate_df[(grad_rate_df['metric']
                                        == 'Four-Year Graduation Rate') &
                                        (grad_rate_df['district']
                                        == 'Canidae School District')]

# Calculate difference from overall grad rate for both start and stop year
years = sorted(grad_rate_df['year'].dropna().unique())
start_year = min(years)
stop_year = max(years)

# Validate: exactly 1 value here
grad_rate_filtered_df[(grad_rate_filtered_df['subgroup']
                                        == 'All Students') &
                                        (grad_rate_filtered_df['year']
                                        == start_year)]['value'].head()

CPU times: total: 78.1 ms
Wall time: 72.8 ms


0    65.103827
Name: value, dtype: float64

In [11]:
# Extract baseline values for group 'All Students'
baseline_start = grad_rate_filtered_df[(grad_rate_filtered_df['subgroup']
                                        == 'All Students') &
                                        (grad_rate_filtered_df['year']
                                        == start_year)]['value'].values[0]
baseline_stop = grad_rate_filtered_df[(grad_rate_filtered_df['subgroup']
                                        == 'All Students') &
                                        (grad_rate_filtered_df['year']
                                        == stop_year)]['value'].values[0]

# Calculate values for disparity
grad_rate_filtered_df = grad_rate_filtered_df[(grad_rate_filtered_df['year']
                                        == start_year) |
                                        (grad_rate_filtered_df['year']
                                        == stop_year)]
grad_rate_filtered_df = grad_rate_filtered_df[~(grad_rate_filtered_df['subgroup'] == 'All Students')]

# Pivot
disparity_df = grad_rate_filtered_df.pivot(index='subgroup', columns='year', values='value').reset_index()
disparity_df['start_year_gap'] = disparity_df[start_year] - baseline_start
disparity_df['stop_year_gap'] = disparity_df[stop_year] - baseline_stop

print(baseline_start)
print(baseline_stop)
disparity_df.head()

65.1038266298446
67.03904067493592


year,subgroup,2020,2025,start_year_gap,stop_year_gap
0,American Indian or Alaska Native,75.972430,79.048968,10.868603,12.009927
1,Asian or Native Hawaiian/Other Pacific Islander,66.343875,71.373827,1.240048,4.334786
2,Black or African American,47.957855,52.798891,-17.145971,-14.240150
3,Economically Disadvantaged,46.331089,46.158530,-18.772737,-20.880511
4,English Language Learner,49.144282,53.168061,-15.959545,-13.870980
